In [4]:
import cmath

def get_twiddle(N, k):
    """Calculates W_N^k = e^(-j * 2*pi*k / N)"""
    angle = -2j * cmath.pi * k / N
    return cmath.exp(angle)

def pack_twiddle_q3_17(w_val):
    """
    Converts a complex float to Q3.17 (20-bit) and 
    packs it into a 40-bit integer [19:0 Imag | 19:0 Real].
    """
    # Scale by 2^17 (131072.0) to give 20 bits total (Q3.17 format)
    real_scaled = int(round(w_val.real * 131072.0))
    imag_scaled = int(round(w_val.imag * 131072.0))
    
    # Mask to 20 bits (Two's Complement)
    real_hex = real_scaled & 0xFFFFF
    imag_hex = imag_scaled & 0xFFFFF

    print(real_hex," ","+ j",imag_hex)
    
    # Pack Imaginary in top 20 bits, Real in bottom 20 bits
    return (imag_hex << 20) | real_hex

def generate_coe(filename, N, num_butterflies):
    """Generates a Vivado-compatible 120-bit COE file for a specific FFT stage."""
    with open(filename, 'w') as f:
        f.write("memory_initialization_radix=16;\n")
        f.write("memory_initialization_vector=\n")
        
        lines = []
        for k in range(num_butterflies):
            # Calculate the 3 non-trivial twiddle factors
            w1 = get_twiddle(N, k)
            w2 = get_twiddle(N, 2*k)
            w3 = get_twiddle(N, 3*k)

            # print("w1 w2 w3: ",w1,w2,w3)
            
            # Pack them into 40-bit chunks
            print("w1:",end=" ")
            pw1 = pack_twiddle_q3_17(w1)
            print("w2:",end=" ")
            pw2 = pack_twiddle_q3_17(w2)
            print("w3:",end=" ")
            pw3 = pack_twiddle_q3_17(w3)
            
            # Combine 3 twiddles into a single 120-bit word
            # Format: W3 (40b) | W2 (40b) | W1 (40b)
            word_120 = (pw3 << 80) | (pw2 << 40) | pw1
            
            # Format as a 30-character hexadecimal string
            lines.append(f"{word_120:030X}")
            
        # Write to file (comma separated, ending with a semicolon)
        f.write(",\n".join(lines))
        f.write(";\n")
    print(f"Generated {filename} ({num_butterflies} addresses).")

if __name__ == "__main__":
    # Stage 1: N=128 (Breaks into 4 chunks of 32) -> 32 unique butterflies
    generate_coe("stage1_twiddles.coe", 128, 32)
    
    # Stage 2: N=32 (Breaks into 4 chunks of 8) -> 8 unique butterflies
    generate_coe("stage2_twiddles.coe", 32, 8)
    
    # Stage 3: N=8 (Breaks into 4 chunks of 2) -> 2 unique butterflies
    generate_coe("stage3_twiddles.coe", 8, 2)

w1: 131072   + j 0
w2: 131072   + j 0
w3: 131072   + j 0
w1: 130914   + j 1042145
w2: 130441   + j 1035729
w3: 129653   + j 1029344
w1: 130441   + j 1035729
w2: 128553   + j 1023005
w3: 125428   + j 1010528
w1: 129653   + j 1029344
w2: 125428   + j 1010528
w3: 118488   + j 992535
w1: 128553   + j 1023005
w2: 121095   + j 998417
w3: 108982   + j 975756
w1: 127144   + j 1016728
w2: 115595   + j 986789
w3: 97118   + j 960553
w1: 125428   + j 1010528
w2: 108982   + j 975756
w3: 83151   + j 947256
w1: 123410   + j 1004419
w2: 101320   + j 965425
w3: 67384   + j 936152
w1: 121095   + j 998417
w2: 92682   + j 955894
w3: 50159   + j 927481
w1: 118488   + j 992535
w2: 83151   + j 947256
w3: 31848   + j 921432
w1: 115595   + j 986789
w2: 72820   + j 939594
w3: 12847   + j 918135
w1: 112424   + j 981192
w2: 61787   + j 932981
w3: 1042145   + j 917662
w1: 108982   + j 975756
w2: 50159   + j 927481
w3: 1023005   + j 920023
w1: 105278   + j 970497
w2: 38048   + j 923148
w3: 1004419   + j 925166
w1: 